In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from agn_subtract import recenter_psf, subtract_agn #, wiener_deconvolve

from astropy.visualization import AsinhStretch, ImageNormalize, make_lupton_rgb, LinearStretch, LogStretch

%matplotlib inline

In [ ]:
def normalize_band(image, vmin=None, vmax=None):
    
    data = image

    if vmin==None or vmax==None:
        vmin = np.percentile(data, 1)
        vmax = np.percentile(data, 99.9)

    norm = ImageNormalize(vmin=vmin, vmax=vmax,
                          #stretch=AsinhStretch(a=asinh_a),
                          #stretch=AsinhStretch(),
                          #stretch=LogStretch(),
                          stretch=LinearStretch(),
                          clip=True,
                         )

    scaled = norm(data)
    return scaled, vmin, vmax


def combine_RGB(R_image, G_image, B_image):
    
    #_, R_vmin, R_vmax = normalize_band(R_image)
    #_, G_vmin, G_vmax = normalize_band(G_image)
    #_, B_vmin, B_vmax = normalize_band(B_image)

    #mean_vmin = np.mean([R_vmin,G_vmin,B_vmin])
    #mean_vmax = np.mean([R_vmax,G_vmax,B_vmax])

    #R_channel, _, _ = normalize_band(R_image, mean_vmin, mean_vmax)
    #G_channel, _, _ = normalize_band(G_image, mean_vmin, mean_vmax)
    #B_channel, _, _ = normalize_band(B_image, mean_vmin, mean_vmax)

    R_channel, _, _ = normalize_band(R_image)
    G_channel, _, _ = normalize_band(G_image)
    B_channel, _, _ = normalize_band(B_image)

    RGB_image = np.dstack([R_channel, G_channel, B_channel])
    
    return RGB_image

    

In [ ]:
def plot_RGB(R_image, G_image, B_image, make_lupton=False, lim=None, positions=None):

    if R_image is None or G_image is None or B_image is None:
        print('Missing data!')
        return 1

    RGB_image = combine_RGB(R_image, G_image, B_image)
    if make_lupton:
        RGB_image = make_lupton_rgb(R_image,
                                    G_image,
                                    B_image,
                                    stretch=0.002, Q=0.001)

    if lim is not None:
        xmin, xmax, ymin, ymax = lim
        RGB_image = RGB_image[xmin:xmax, ymin:ymax]
    
    fig = plt.figure(figsize=(6,6))
    im = plt.imshow(RGB_image,
                    origin='lower')

    if positions is not None:
        x_arr = positions[:,0]
        y_arr = positions[:,1]
        plt.plot(x_arr, y_arr, 'w*')

    return 0

In [ ]:
def run(
    band,
    folder="../",
    positions=((27., 27.), (22., 19.)),
    lens_mask_radius=4.,
    variance_percentile=60,
    max_position_shift=3.5,
):
    """
    Fit and subtract AGN point sources for one band.

    Parameters
    ----------
    band : str
        Band name, e.g. 'g', 'r', 'i'.

    folder : str
        Folder containing cutout_<band>.npz.

    positions : sequence of (x, y)
        Initial AGN positions.

    lens_mask_radius : float
        Radius of the circular region masked around the lens center
        during the AGN fit.

    variance_percentile : float
        Percentile used to cap the variance during AGN fitting.

    max_position_shift : float
        Maximum allowed AGN position shift from the initial positions.

    Returns
    -------
    result : dict
        Output from subtract_agn().

    residual : 2D ndarray
        AGN-subtracted image.

    chi : 2D ndarray
        Residual divided by the original noise sigma.
    """

    # --------------------------------------------------
    # Load data
    # --------------------------------------------------

    filename = f"{folder}/cutout_{band}.npz"
    cutout = np.load(filename)

    image = cutout["image"]
    variance = cutout["variance"]
    psf = cutout["psf"]

    # Currently not used
    # mask = cutout["mask"][:, :, 1]

    positions = np.asarray(positions, dtype=float)

    # --------------------------------------------------
    # Recenter PSF
    # --------------------------------------------------

    psf = recenter_psf(psf)

    # --------------------------------------------------
    # Mask the approximate lens position
    # --------------------------------------------------

    x_lens, y_lens = positions.mean(axis=0)

    print(f"Lens center: ({x_lens:.2f}, {y_lens:.2f})")

    yy, xx = np.indices(image.shape)

    lens_mask = (
        (xx - x_lens)**2 +
        (yy - y_lens)**2
    ) < lens_mask_radius**2

    # --------------------------------------------------
    # Variance used only for AGN fitting
    # --------------------------------------------------

    fit_variance = variance.copy()

    good = np.isfinite(variance) & (variance > 0)

    v_cap = np.percentile(
        variance[good],
        variance_percentile,
    )

    fit_variance[good] = np.minimum(
        fit_variance[good],
        v_cap,
    )

    # --------------------------------------------------
    # Fit and subtract AGNs
    # --------------------------------------------------

    result = subtract_agn(
        image=image,
        variance=fit_variance,
        psf=psf,
        positions=positions,
        bad_mask=lens_mask,
        max_position_shift=max_position_shift,
    )

    agn_model = result["model"]
    residual = result["residual"]
    fitted_positions = result["positions"]

    # Use the ORIGINAL variance for chi
    chi = residual / np.sqrt(variance)

    # --------------------------------------------------
    # Print fit results
    # --------------------------------------------------

    print("\nFitted fluxes:")
    print(result["fluxes"])

    print("\nInitial positions:")
    print(positions)

    print("\nFitted positions:")
    print(fitted_positions)

    print("\nPosition shifts:")
    print(fitted_positions - positions)

    print("\nChi at fitted AGN positions:")

    for i, (x, y) in enumerate(fitted_positions):
        ix = int(round(x))
        iy = int(round(y))

        print(
            f"{i}: "
            f"position=({x:.3f}, {y:.3f}), "
            f"chi={chi[iy, ix]:.2f}"
        )

    # --------------------------------------------------
    # Plot image, model, residual
    # --------------------------------------------------

    fig, axes = plt.subplots(
        1, 3,
        figsize=(12, 4),
    )

    vmin, vmax = np.percentile(image, [1, 99])
    rvmin, rvmax = np.percentile(residual, [1, 99])

    im = axes[0].imshow(
        image,
        origin="lower",
        vmin=vmin,
        vmax=vmax,
    )
    axes[0].set_title("Original")
    fig.colorbar(im, ax=axes[0])

    im = axes[1].imshow(
        agn_model,
        origin="lower",
        vmin=vmin,
        vmax=vmax,
    )
    axes[1].set_title("AGN model")
    fig.colorbar(im, ax=axes[1])

    im = axes[2].imshow(
        residual,
        origin="lower",
        vmin=rvmin,
        vmax=rvmax,
    )
    axes[2].set_title("AGN-subtracted")
    fig.colorbar(im, ax=axes[2])

    for ax in axes:
        ax.plot(
            fitted_positions[:, 0],
            fitted_positions[:, 1],
            "r+",
        )
        ax.set_xticks([])
        ax.set_yticks([])

    plt.tight_layout()
    plt.show()

    # --------------------------------------------------
    # Plot chi residual
    # --------------------------------------------------

    plt.figure(figsize=(5, 4))

    plt.imshow(
        chi,
        origin="lower",
        vmin=-5,
        vmax=5,
        cmap="bwr",
    )

    plt.plot(
        fitted_positions[:, 0],
        fitted_positions[:, 1],
        "k+",
    )

    plt.colorbar(
        label=r"Residual / $\sigma$"
    )

    plt.title(f"{band}-band residual significance")
    plt.tight_layout()
    plt.show()

    return result, residual, chi

In [ ]:
result_u, residual_u, chi_u = run("u")
result_g, residual_g, chi_g = run("g")
result_r, residual_r, chi_r = run("r")
result_i, residual_i, chi_i = run("i")
result_z, residual_z, chi_z = run("z")

In [ ]:
plot_RGB(residual_r, residual_g, residual_u, positions=result_g['positions'])
plot_RGB(chi_r, chi_g, chi_u, positions=result_g['positions'])

In [ ]:
plot_RGB(residual_i, residual_r, residual_g, positions=result_r['positions'])
plot_RGB(chi_i, chi_r, chi_g, positions=result_r['positions'])

In [ ]:
plot_RGB(residual_z, residual_i, residual_r, positions=result_i['positions'])
plot_RGB(chi_z, chi_i, chi_r, positions=result_i['positions'])